In [2]:
import numpy as np
from pygmid import Lookup as lk
from scipy.interpolate import interp1d

In [3]:
# Load technology data
NCH = lk('../../sky130_lookup/simulation/nfet_01v8.mat')
PCH = lk('../../sky130_lookup/simulation/pfet_01v8.mat')

# ===================
# 5T-OTA DESIGN SPECS
# ===================
VDD = 1.8  # Supply Voltage (V)
CL  = 2e-12  # Load Capacitance (F)
SR_spec = 1e6  # Slew Rate (V/s)
GBW_spec = 1e6  # Gain Bandwidth (Hz)
Gain_spec_dB = 38  # DC Gain (dB)
Gain_spec = 10**(Gain_spec_dB/20)  # Convert dB to V/V
PM_spec = 65  # Phase Margin (degrees)
Power_spec = 10e-6  # Power Consumption (W)

In [4]:
# ===============================
# LUT FOR DIODE CONNECTED DEVICES
# ===============================
vgs_sweep = np.arange(0.05, VDD+0.1, 0.01)

# Function to create diode-connected LUT
def diode_connected_lut(device_data, vgs_sweep):
    L_values = np.unique(device_data['L'])
    diode_lut = {}

    for l_val in L_values:
        gm_id = device_data.lookup('GM_ID', L=l_val, VGS=vgs_sweep, VDS=vgs_sweep, VSB=0)
        diode_lut[l_val] = np.diag(gm_id)

    return diode_lut

# Create LUTs for diode-connected NMOS and PMOS
nch_results = diode_connected_lut(NCH, vgs_sweep)
pch_results = diode_connected_lut(PCH, vgs_sweep)

# Function to get VGS for a target gm/ID
def getVGS_diode(device_type, target_gm_id, length):
    if device_type.lower() == 'nmos':
        gm_id_vec = nch_results[length]
    elif device_type.lower() == 'pmos':
        gm_id_vec = pch_results[length]
    else:
        raise ValueError("Device type must be 'nmos' or 'pmos'.")

    get_vgs = interp1d(gm_id_vec, vgs_sweep, kind='linear', bounds_error=False)
    vgs_required = get_vgs(target_gm_id)
    return vgs_required

In [5]:
def get_W(gm1, gm2, L_1, L_2, ID):
    gm_ID_1 = gm1 / ID
    gm_ID_2 = gm2 / ID

    # Vgs_2 = brentq(vgs_error, 0.3, 1.8, args=(gm_ID_2, L_2, 0))
    Vgs_2 = getVGS_diode('PMOS', gm_ID_2, L_2)

    JD_1 = NCH.lookup('ID_W', GM_ID=gm_ID_1, VDS=VDD/2, VSB=0, L=L_1)
    JD_2 = PCH.lookup('ID_W', GM_ID=gm_ID_2, VDS=Vgs_2, VSB=0, L=L_2)

    W_1 = ID / JD_1
    W_2 = ID / JD_2

    return W_1, W_2

In [6]:
def get_specVars(gm1, gm2, L_1, L_2, ID):
    gm_ID_1 = gm1 / ID
    gm_ID_2 = gm2 / ID

    # Vgs_2 = brentq(vgs_error, 0.3, 1.8, args=(gm_ID_2, L_2, 0))
    Vgs_2 = getVGS_diode('PMOS', gm_ID_2, L_2)

    Cdd_1 = gm1 / NCH.lookup('GM_CDD', GM_ID=gm_ID_1, VDS=VDD/2, VSB=0, L=L_1)
    Cdd_2 = gm2 / PCH.lookup('GM_CDD', GM_ID=gm_ID_2, VDS=Vgs_2, VSB=0, L=L_2)
    Cpar = Cdd_1 + Cdd_2

    Cgg_2 = gm2 / PCH.lookup('GM_CGG', GM_ID=gm_ID_2, VDS=Vgs_2, VSB=0, L=L_2)
    Cx = Cpar + Cgg_2

    JD_1 = NCH.lookup('ID_W', GM_ID=gm_ID_1, VDS=VDD/2, VSB=0, L=L_1)
    JD_2 = PCH.lookup('ID_W', GM_ID=gm_ID_2, VDS=Vgs_2, VSB=0, L=L_2)

    W_1 = ID / JD_1
    W_2 = ID / JD_2

    gds_1 = gm1 / NCH.lookup('GM_GDS', GM_ID=gm_ID_1, VDS=VDD/2, VSB=0, L=L_1)
    gds_2 = gm2 / PCH.lookup('GM_GDS', GM_ID=gm_ID_2, VDS=Vgs_2, VSB=0, L=L_2)

    return Cx, Cpar, W_1, W_2, gds_1, gds_2

In [7]:
def get_feasRegion(gm_ID, L_1, L_2, SR_spec, CL, Power_spec, GBW_spec):
    gm_ID_min = gm_ID [0]
    gm_ID_max = gm_ID[1]

    ID_min = SR_spec * CL / 2  # Minimum current (A)
    ID_max = np.floor(Power_spec * 1e7 / VDD) / 2 * 1e7  # Maximum tail current (A)

    gm1_min_GBW = 2 * np.pi * GBW_spec * CL  # Minimum gm1 for GBW spec
    gm1_min_gmID = gm_ID_min * ID_min  # Minimum gm1 for gm/ID
    gm1_min = max(gm1_min_GBW, gm1_min_gmID)  # Overall minimum gm1
    gm1_max = gm_ID_max * ID_max  # Maximum gm1 for gm/ID

    gm2_min = gm_ID_min * ID_min  # Minimum gm2 for gm/ID
    gm2_max = gm_ID_max * ID_max  # Maximum gm2 for gm/ID

    L_1_min = L_1 [1]
    L_1_max = L_1 [0]

    L_2_min = L_2 [1]
    L_2_max = L_2 [0]

    return gm1_min, gm1_max, gm2_min, gm2_max, L_1_min, L_1_max, L_2_min, L_2_max, ID_min, ID_max

In [8]:
def update_feasRegion(gm1_min, gm1_max, gm2_min, gm2_max, L_1_min, L_1_max, L_2_min, L_2_max, ID_min, ID_max):
    #  TBA
    return None

In [9]:
gm_ID = (3, 20)  # gm/ID range (S/A)
L_1 = (0.15, 3)  # L1 range (μm)
L_2 = (0.15, 3)  # L2 range (μm)

gm1_min, gm1_max, gm2_min, gm2_max, L_1_min, L_1_max, L_2_min, L_2_max, ID_min, ID_max = get_feasRegion(gm_ID, L_1, L_2, SR_spec, CL, Power_spec, GBW_spec)

In [10]:
gm_ID_1 = 18  # S/A
gm_ID_2 = 6  # S/A
L_1 = 0.5  # um
L_2 = 3  # um
ID = 2e-6  # A

gm1 = gm_ID_1*ID
gm2 = gm_ID_2*ID

Cx, Cpar, W_1, W_2, gds_1, gds_2 = get_specVars(gm1, gm2, L_1=L_1, L_2=L_2, ID=ID)

In [11]:
# ==================
# SPECS VERIFICATION
# ==================

Cload_total = CL + Cpar

# Width Verification
if W_1 >= 0.42 or W_2 >= 0.42 :
    print(f"Transistor Widths : W1 = {W_1:.2f}μm, W2 = {W_2:.2f}μm")

# Slew Rate Verification
SR_calc = (ID * 2) / Cload_total
if SR_calc >= SR_spec:
    print(f"Slew Rate : {SR_calc*1e-6:.2f}V/μs")

# Gain Bandwidth Verification
GBW_calc = gm1 / (2 * np.pi * Cload_total)
if GBW_calc >= GBW_spec:
    print(f"Gain Bandwidth : {GBW_calc*1e-6:.2f}MHz")

# Gain Verification
Gain_calc = gm1 / (gds_1 + gds_2)
if Gain_calc >= Gain_spec:
    print(f"Gain : {20*np.log10(Gain_calc):.2f}dB")

# Phase Margin (degrees) Verification
PM_calc = 90 - (np.arctan(GBW_calc / (gm2 / Cx))*(180/np.pi))
if PM_calc >= PM_spec:
    print(f"Phase Margin : {PM_calc:.2f} degrees")

# Power Consumption Verification
Power_calc = ID * 2 * VDD
if Power_calc <= Power_spec:
    print(f"Power Consumption : {Power_calc*1e6:.2f}μW")

# Active area calculation
Area_active = 2 * ((W_1 * L_1) + (W_2 * L_2))
print(f"\nTotal Active Area of 5T-OTA : {Area_active:.2f}μm²")

Transistor Widths : W1 = 1.45μm, W2 = 1.70μm
Slew Rate : 2.00V/μs
Gain Bandwidth : 2.86MHz
Gain : 42.12dB
Phase Margin : 89.60 degrees
Power Consumption : 7.20μW

Total Active Area of 5T-OTA : 11.64μm²
